# AI Shield Deployment Artifact Downloader

This notebook is only for packaging saved model artifacts from Google Drive.

It does **not** train anything. It does **not** rerun EfficientNet, VAE, OpenForensics, ViT, or fusion. It only:

1. mounts Google Drive,
2. finds the saved artifacts from Phase 1, Phase 2, and Phase 3,
3. copies the needed files into a clean deployment folder,
4. writes a manifest and feature schema,
5. zips the folder so I can download it and place it into the repo for deployment.

The output zip should be downloaded from Colab and added to the repo later.

## 1. Mount Drive

This gives the notebook access to the saved AI Shield folders:

```text
/content/drive/MyDrive/AI_Shield_Phase1
/content/drive/MyDrive/AI_Shield_Phase2_Semantic
/content/drive/MyDrive/AI_Shield_Phase3_Fusion
```

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
print('Drive root:', DRIVE_ROOT)
print('Exists:', DRIVE_ROOT.exists())

## 2. Config

These are the artifact roots created by the three project notebooks. If I renamed a Drive folder, this is the only place I should edit.

In [ ]:
from dataclasses import dataclass, asdict
import json
import os
import shutil
import hashlib
from datetime import datetime

@dataclass
class BundleConfig:
    drive_root: str = '/content/drive/MyDrive'
    phase1_dirname: str = 'AI_Shield_Phase1'
    phase2_dirname: str = 'AI_Shield_Phase2_Semantic'
    phase2_version: str = 'semantic_vit_transformer_attention_v1'
    phase3_dirname: str = 'AI_Shield_Phase3_Fusion'
    phase3_version: str = 'forensic_semantic_glm_v1'
    output_dir: str = '/content/ai_shield_deployment_bundle'
    output_zip: str = '/content/ai_shield_deployment_bundle.zip'
    download_zip_at_end: bool = True

CONFIG = BundleConfig()
print(json.dumps(asdict(CONFIG), indent=2))

## 3. Helper Functions

The logic here is intentionally conservative. Required files must exist. Optional files are copied if available and skipped if missing. For OpenForensics, the notebook searches Drive because I may have uploaded the `.json` and `.h5` to a different folder.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def newest_matching(root, pattern):
    root = Path(root)
    matches = [p for p in root.rglob(pattern) if p.is_file()]
    if not matches:
        return None
    return max(matches, key=lambda p: p.stat().st_mtime)


def copy_artifact(src, dst, required=True, label=None):
    src = Path(src) if src is not None else None
    dst = Path(dst)
    label = label or dst.name

    if src is None or not src.exists():
        if required:
            raise FileNotFoundError(f'Required artifact missing for {label}: {src}')
        print(f'[optional missing] {label}: {src}')
        return None

    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    record = {
        'label': label,
        'source': str(src),
        'destination': str(dst),
        'size_bytes': int(dst.stat().st_size),
        'sha256': sha256_file(dst),
        'modified_source': datetime.fromtimestamp(src.stat().st_mtime).isoformat(),
    }
    print(f'[copied] {label}: {src} -> {dst} ({record["size_bytes"] / 1024 / 1024:.2f} MB)')
    return record


def write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding='utf-8')
    print('wrote', path)

## 4. Define the Deployment Artifact List

The deployment app will eventually need the actual model checkpoints plus the final GLM. I also copy metrics/config files because they are useful for a model card and sanity checks.

In [ ]:
drive_root = Path(CONFIG.drive_root)
phase1 = drive_root / CONFIG.phase1_dirname
phase2 = drive_root / CONFIG.phase2_dirname / CONFIG.phase2_version
phase3 = drive_root / CONFIG.phase3_dirname / CONFIG.phase3_version

effnet_exp = 'effnet_b0_tinygenimage_full_balanced_v1'
vae_exp = 'vae_real_only_tinygenimage_v1'

bundle_root = Path(CONFIG.output_dir)
models_root = bundle_root / 'models'

# Clean only the temporary Colab bundle folder, not Drive.
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True, exist_ok=True)

required_artifacts = [
    # Phase 1 EfficientNet broad forensic branch
    ('phase1_effnet_best_checkpoint', phase1 / effnet_exp / f'{effnet_exp}_best.pt', models_root / 'phase1_effnet' / f'{effnet_exp}_best.pt'),
    ('phase1_effnet_platt_calibrator', phase1 / effnet_exp / 'platt_calibrator.pkl', models_root / 'phase1_effnet' / 'platt_calibrator.pkl'),

    # Phase 1 VAE anomaly branch
    ('phase1_vae_best_checkpoint', phase1 / vae_exp / f'{vae_exp}_best.pt', models_root / 'phase1_vae' / f'{vae_exp}_best.pt'),
    ('phase1_vae_reconstruction_errors', phase1 / vae_exp / 'vae_reconstruction_errors.npz', models_root / 'phase1_vae' / 'vae_reconstruction_errors.npz'),

    # Phase 2 semantic branch
    ('phase2_semantic_best_checkpoint', phase2 / 'checkpoints' / 'semantic_vit_transformer_attention_best.pt', models_root / 'phase2_semantic' / 'semantic_vit_transformer_attention_best.pt'),

    # Phase 3 final GLM
    ('phase3_full_glm', phase3 / 'checkpoints' / 'phase3_full_forensic_semantic_glm_v1.pkl', models_root / 'phase3_fusion' / 'phase3_full_forensic_semantic_glm_v1.pkl'),
]

optional_artifacts = [
    # Phase 1 EfficientNet useful metadata
    ('phase1_effnet_config', phase1 / effnet_exp / f'{effnet_exp}_config.json', models_root / 'phase1_effnet' / f'{effnet_exp}_config.json'),
    ('phase1_effnet_history', phase1 / effnet_exp / f'{effnet_exp}_history.json', models_root / 'phase1_effnet' / f'{effnet_exp}_history.json'),
    ('phase1_effnet_calibration_metrics', phase1 / effnet_exp / 'calibration_metrics.json', models_root / 'phase1_effnet' / 'calibration_metrics.json'),
    ('phase1_effnet_threshold_sweep', phase1 / effnet_exp / 'threshold_sweep.csv', models_root / 'phase1_effnet' / 'threshold_sweep.csv'),

    # Phase 1 VAE useful metadata
    ('phase1_vae_config', phase1 / vae_exp / f'{vae_exp}_config.json', models_root / 'phase1_vae' / f'{vae_exp}_config.json'),
    ('phase1_vae_history', phase1 / vae_exp / f'{vae_exp}_history.json', models_root / 'phase1_vae' / f'{vae_exp}_history.json'),
    ('phase1_vae_last_checkpoint', phase1 / vae_exp / f'{vae_exp}_last.pt', models_root / 'phase1_vae' / f'{vae_exp}_last.pt'),

    # Phase 1 late-fusion model card artifacts
    ('phase1_branch_outputs_cache', phase1 / 'late_fusion_face_gated_v1' / 'cache' / 'branch_outputs_v1.csv', models_root / 'phase1_late_fusion' / 'branch_outputs_v1.csv'),
    ('phase1_fusion_metrics', phase1 / 'late_fusion_face_gated_v1' / 'results' / 'fusion_logistic_metrics_v1.json', models_root / 'phase1_late_fusion' / 'fusion_logistic_metrics_v1.json'),

    # Phase 2 model card artifacts
    ('phase2_semantic_metrics', phase2 / 'results' / 'semantic_vit_transformer_attention_metrics_v1.json', models_root / 'phase2_semantic' / 'semantic_vit_transformer_attention_metrics_v1.json'),
    ('phase2_semantic_cache', phase2 / 'cache' / 'semantic_branch_outputs_v1.csv', models_root / 'phase2_semantic' / 'semantic_branch_outputs_v1.csv'),

    # Phase 3 final fusion artifacts
    ('phase3_forensic_only_glm', phase3 / 'checkpoints' / 'phase3_forensic_only_glm_v1.pkl', models_root / 'phase3_fusion' / 'phase3_forensic_only_glm_v1.pkl'),
    ('phase3_metrics', phase3 / 'results' / 'phase3_fusion_metrics_v1.json', models_root / 'phase3_fusion' / 'phase3_fusion_metrics_v1.json'),
    ('phase3_coefficients', phase3 / 'results' / 'phase3_full_glm_coefficients_v1.csv', models_root / 'phase3_fusion' / 'phase3_full_glm_coefficients_v1.csv'),
    ('phase3_threshold_sweeps', phase3 / 'results' / 'phase3_threshold_sweeps_v1.csv', models_root / 'phase3_fusion' / 'phase3_threshold_sweeps_v1.csv'),
    ('phase3_split_manifest', phase3 / 'manifests' / 'phase3_fusion_split_manifest_v1.json', models_root / 'phase3_fusion' / 'phase3_fusion_split_manifest_v1.json'),
]

print('Phase 1 root:', phase1, phase1.exists())
print('Phase 2 root:', phase2, phase2.exists())
print('Phase 3 root:', phase3, phase3.exists())
print('Required artifacts:', len(required_artifacts))
print('Optional artifacts:', len(optional_artifacts))

## 5. Find OpenForensics Keras Files

The OpenForensics model may be in the repo folder or somewhere in Drive. This cell searches Drive for `dffnetv2B0.json`, `dffnetv2B0.h5`, and optionally the zip.

In [ ]:
openforensics_json = newest_matching(drive_root, 'dffnetv2B0.json')
openforensics_h5 = newest_matching(drive_root, 'dffnetv2B0.h5')
openforensics_zip = newest_matching(drive_root, 'dffnetv2B0.zip')

print('OpenForensics JSON:', openforensics_json)
print('OpenForensics H5:', openforensics_h5)
print('OpenForensics ZIP:', openforensics_zip)

if openforensics_json is None or openforensics_h5 is None:
    raise FileNotFoundError(
        'Could not find dffnetv2B0.json and dffnetv2B0.h5 in Drive. '
        'Upload them to Drive first, then rerun this notebook.'
    )

## 6. Copy Artifacts Into a Deployment Folder

Required files stop the notebook if missing. Optional files are useful, but the bundle can still be created without them.

In [ ]:
manifest = {
    'created_at': datetime.utcnow().isoformat() + 'Z',
    'config': asdict(CONFIG),
    'required': [],
    'optional': [],
    'notes': [
        'This bundle contains saved AI Shield artifacts for deployment.',
        'It does not contain the original training datasets.',
        'It was produced from Google Drive artifacts without retraining.',
    ],
}

for label, src, dst in required_artifacts:
    manifest['required'].append(copy_artifact(src, dst, required=True, label=label))

for label, src, dst in optional_artifacts:
    rec = copy_artifact(src, dst, required=False, label=label)
    if rec is not None:
        manifest['optional'].append(rec)

# OpenForensics is required for the full final feature pipeline.
manifest['required'].append(copy_artifact(openforensics_json, models_root / 'openforensics' / 'dffnetv2B0.json', required=True, label='openforensics_keras_json'))
manifest['required'].append(copy_artifact(openforensics_h5, models_root / 'openforensics' / 'dffnetv2B0.h5', required=True, label='openforensics_keras_h5'))
if openforensics_zip is not None:
    rec = copy_artifact(openforensics_zip, models_root / 'openforensics' / 'dffnetv2B0.zip', required=False, label='openforensics_keras_zip')
    if rec is not None:
        manifest['optional'].append(rec)

write_json(manifest, bundle_root / 'artifact_manifest.json')

## 7. Write Deployment Feature Schema

This schema records the exact feature order expected by the Phase 3 GLM. The deployment app should compute these columns for a new image, then pass them to `phase3_full_forensic_semantic_glm_v1.pkl` in this order.

In [ ]:
FORNSIC_FEATURES = [
    'effnet_logit',
    'effnet_prob',
    'effnet_calibrated_prob',
    'vae_recon_error',
    'vae_recon_percentile',
    'face_present',
    'num_faces',
    'largest_face_ratio',
    'face_confidence',
    'openforensics_applicable',
    'openforensics_prob_fake',
]

SEMANTIC_FEATURES = [
    'semantic_vit_logit',
    'semantic_vit_prob',
    'semantic_attention_entropy',
    'semantic_attention_focus',
    'semantic_self_attention_entropy',
    'semantic_self_attention_focus',
]

schema = {
    'label_direction': '1 means fake, 0 means real',
    'final_model': 'models/phase3_fusion/phase3_full_forensic_semantic_glm_v1.pkl',
    'recommended_threshold_from_phase3_validation': 0.42,
    'forensic_features': FORNSIC_FEATURES,
    'semantic_features': SEMANTIC_FEATURES,
    'full_feature_order': FORNSIC_FEATURES + SEMANTIC_FEATURES,
    'notes': [
        'The GLM expects a pandas DataFrame with these feature names.',
        'OpenForensics old Keras score direction is p(real); deployment must convert to p(fake)=1-p(real).',
        'If no valid face is present, use openforensics_applicable=0 and openforensics_prob_fake=0.5.',
    ],
}

write_json(schema, bundle_root / 'feature_schema.json')
print(json.dumps(schema, indent=2))

## 8. Add a README for the Deployment Bundle

In [ ]:
readme = '''# AI Shield Deployment Artifact Bundle

This folder was generated from Google Drive artifacts by `make_ai_shield_deployment_bundle_colab.ipynb`.

It contains saved model artifacts only. It does not contain the original datasets and it does not represent retraining.

## Folder layout

```text
models/
  phase1_effnet/
  phase1_vae/
  openforensics/
  phase2_semantic/
  phase3_fusion/
artifact_manifest.json
feature_schema.json
```

## Inference order

```text
image
-> Phase 1 EfficientNet features
-> Phase 1 VAE reconstruction features
-> FaceGate + OpenForensics features
-> Phase 2 semantic ViT attention features
-> Phase 3 GLM final fake probability
```

The final GLM expects the feature order written in `feature_schema.json`.
'''

(bundle_root / 'README.md').write_text(readme, encoding='utf-8')
print((bundle_root / 'README.md').read_text())

## 9. Zip and Download

After this runs, download `ai_shield_deployment_bundle.zip` from Colab and put it into the local repo. Then we can build the deployment app around it.

In [ ]:
output_zip = Path(CONFIG.output_zip)
if output_zip.exists():
    output_zip.unlink()

shutil.make_archive(str(output_zip.with_suffix('')), 'zip', root_dir=bundle_root)

print('Created zip:', output_zip)
print('Zip size MB:', output_zip.stat().st_size / 1024 / 1024)

print('\nBundle tree:')
for path in sorted(bundle_root.rglob('*')):
    rel = path.relative_to(bundle_root)
    if path.is_file():
        print(f'  {rel} ({path.stat().st_size / 1024 / 1024:.2f} MB)')
    else:
        print(f'  {rel}/')

if CONFIG.download_zip_at_end:
    from google.colab import files
    files.download(str(output_zip))